# Ficha de trabalho 16 — Tratamento de dados de várias fontes (resolução)

Cinco exercícios, do mais simples ao mais completo, e uma atividade de extensão. Cada célula cria os dados de que precisa.

## Exercício 1 — ficheiro de texto

Lê `inscritos.txt` (um nome por linha), guarda os nomes numa lista sem a mudança de linha, imprime quantos inscritos há e cada nome precedido de um travessão.

**Extensão:** acrescenta um quinto nome e confirma que passa a contar 5.

In [1]:
with open("inscritos.txt", "w", encoding="utf-8") as f:
    f.write("Ana Silva\nBruno Costa\nCarla Mendes\nDuarte Pinto\n")

with open("inscritos.txt", encoding="utf-8") as f:
    nomes = [linha.strip() for linha in f]
print(f"Inscritos: {len(nomes)}")
for nome in nomes:
    print("-", nome)

Inscritos: 4
- Ana Silva
- Bruno Costa
- Carla Mendes
- Duarte Pinto


In [2]:
# Extensão
with open("inscritos.txt", "a", encoding="utf-8") as f:
    f.write("Eva Rocha\n")
with open("inscritos.txt", encoding="utf-8") as f:
    print(f"Inscritos: {len([l for l in f])}")

Inscritos: 5


## Exercício 2 — ficheiro CSV

Lê `stock.csv` com `csv.DictReader` e imprime o nome e o preço do produto mais caro.

**Extensão:** imprime também o produto com menos unidades em stock.

In [3]:
import csv
with open("stock.csv", "w", encoding="utf-8") as f:
    f.write("produto,quantidade,preco\nCaneta,50,0.60\nCaderno,30,2.50\nBorracha,80,0.30\nMochila,12,15.00\n")

with open("stock.csv", encoding="utf-8") as f:
    produtos = list(csv.DictReader(f))
mais_caro = max(produtos, key=lambda p: float(p["preco"]))
print(f"Produto mais caro: {mais_caro['produto']} ({float(mais_caro['preco']):.2f} €)")
# Extensão
menos = min(produtos, key=lambda p: int(p["quantidade"]))
print(f"Menos unidades: {menos['produto']} ({menos['quantidade']})")

Produto mais caro: Mochila (15.00 €)
Menos unidades: Mochila (12)


## Exercício 3 — base de dados SQLite

Cria `cinema.db` com a tabela `filmes` e lista só os títulos com classificação igual ou superior a 8, do mais bem classificado para o menos.

**Extensão:** insere um filme novo e confirma que aparece na consulta, se tiver 8 ou mais.

In [4]:
import sqlite3
ligacao = sqlite3.connect("cinema.db")
cursor = ligacao.cursor()
cursor.execute("CREATE TABLE IF NOT EXISTS filmes (titulo TEXT, classificacao REAL)")
cursor.execute("DELETE FROM filmes")
filmes = [("Matriz", 8.7), ("Robô Desastrado", 6.2), ("Viagem ao Fundo do Código", 9.1), ("Loop Infinito", 7.5)]
cursor.executemany("INSERT INTO filmes VALUES (?, ?)", filmes)
ligacao.commit()

cursor.execute("SELECT titulo FROM filmes WHERE classificacao >= ? ORDER BY classificacao DESC", (8,))
for (titulo,) in cursor.fetchall():
    print(titulo)

# Extensão
cursor.execute("INSERT INTO filmes VALUES (?, ?)", ("O Regresso do Código", 8.2))
ligacao.commit()
print("--- depois de inserir ---")
cursor.execute("SELECT titulo FROM filmes WHERE classificacao >= ? ORDER BY classificacao DESC", (8,))
for (titulo,) in cursor.fetchall():
    print(titulo)
ligacao.close()

Viagem ao Fundo do Código
Matriz
--- depois de inserir ---
Viagem ao Fundo do Código
Matriz
O Regresso do Código


## Exercício 4 — folha de cálculo XLSX

Lê `notas.xlsx` e calcula a média da turma.

**Extensão:** imprime quantos alunos têm nota igual ou superior à média.

In [5]:
# Nota: o openpyxl não vem com o Python. Se o import falhar, instala-o no terminal do VS Code:
#   pip install openpyxl
#   (sem permissões de administrador: pip install --user openpyxl)
import openpyxl
wb = openpyxl.Workbook(); ws = wb.active; ws.title = "Notas"
ws.append(["nome", "nota"])
for linha in [("Ana", 16), ("Bruno", 12), ("Carla", 18), ("Duarte", 9)]:
    ws.append(linha)
wb.save("notas.xlsx")

wb = openpyxl.load_workbook("notas.xlsx")
ws = wb.active
notas = [linha[1] for linha in ws.iter_rows(min_row=2, values_only=True)]
media = sum(notas) / len(notas)
print(f"Média da turma: {media:.2f}")
# Extensão
print(f"Alunos com nota >= média: {len([n for n in notas if n >= media])}")

Média da turma: 13.75
Alunos com nota >= média: 2


## Exercício 5 — função geradora

`multiplos_de(base, limite)` dá os múltiplos de `base` até `limite`.

**Extensão:** soma de todos os múltiplos de 5 até 100.

In [6]:
def multiplos_de(base, limite):
    n = base
    while n <= limite:
        yield n
        n += base

for m in multiplos_de(3, 20):
    print(m)
# Extensão
print(sum(multiplos_de(5, 100)))

3
6
9
12
15
18
1050


## Atividade de extensão — resumo de vendas

Lê `vendas.csv`, grava cada linha numa tabela SQLite, calcula com `GROUP BY` o total de unidades e o valor por produto e escreve `resumo_vendas.xlsx`.

In [7]:
# Nota: o openpyxl não vem com o Python. Se o import falhar, instala-o no terminal do VS Code:
#   pip install openpyxl
#   (sem permissões de administrador: pip install --user openpyxl)
import csv, sqlite3, openpyxl
with open("vendas.csv", "w", encoding="utf-8") as f:
    f.write("produto,quantidade,preco\nCaneta,5,0.60\nCaderno,2,2.50\nCaneta,3,0.60\nBorracha,10,0.30\nCaderno,1,2.50\n")

ligacao = sqlite3.connect(":memory:")
cursor = ligacao.cursor()
cursor.execute("CREATE TABLE vendas (produto TEXT, quantidade INTEGER, preco REAL)")
with open("vendas.csv", encoding="utf-8") as f:
    for linha in csv.DictReader(f):
        cursor.execute("INSERT INTO vendas VALUES (?, ?, ?)",
                       (linha["produto"], int(linha["quantidade"]), float(linha["preco"])))
ligacao.commit()
cursor.execute("SELECT produto, SUM(quantidade), SUM(quantidade * preco) FROM vendas GROUP BY produto")
resumo = cursor.fetchall()
ligacao.close()

relatorio = openpyxl.Workbook(); ws = relatorio.active; ws.title = "Resumo de vendas"
ws.append(["produto", "unidades vendidas", "valor total"])
for produto, unidades, valor in resumo:
    ws.append([produto, unidades, round(valor, 2)])
relatorio.save("resumo_vendas.xlsx")

for linha in openpyxl.load_workbook("resumo_vendas.xlsx").active.iter_rows(values_only=True):
    print(linha)

('produto', 'unidades vendidas', 'valor total')
('Borracha', 10, 3)
('Caderno', 3, 7.5)
('Caneta', 8, 4.8)
